# Recovering LLMGPR's pipeline — catalogue radius x history accounting

`output5.ipynb` found the first thing that ever hit their `#POIs`: read as a **region
catalogue** at R = 10 km around Yang's city centres it comes to **82,188 against their 80,962 —
1.5% off**, with users at 0.94x. Only the check-in column missed, and it missed with a shape:

```
                       R=10, Tu>=30      theirs     ratio
users                         7,051       7,507     0.94x
POIs (catalogue)             82,188      80,962     1.02x
check-ins (in-region)       574,546   1,214,631     0.47x
check-ins per user             81.5       161.8     1.99x
```

**1.99x is not a tuning gap.** It is what you get if users are *selected* by their activity
inside the region but *counted* over their full histories — a city core holds roughly half of
a heavy urban user's check-ins. That is also the natural way to build this: the region defines
the POI catalogue you rank against, while the sequence you feed the model is the user's own.

So this notebook crosses two axes their paper never distinguishes:

- **radius** — one shared R, then a per-city grid (motivated: their LA/Chicago slices are ~4x
  under-weight relative to NYC, so their regions are probably not the same size)
- **accounting** — is `#check-ins` in-region only, or the selected users' full histories? and is
  the user threshold measured in-region or on the full history?

It also fixes a measurement error in test D. That printed `cats` over the *filtered check-ins*,
not the catalogue. Under the catalogue reading `#categories` must be counted over the catalogue
too — and this is the discriminating test: **436 was measured over the full bounding box.** If a
10 km catalogue carries fewer, the category match that identified the dataset belongs to the
wide region and pulls against the narrow one. Either way we learn which column to trust.

Cheap — reuses `llmgpr_checkins_A.parquet` and `llmgpr_pois_xy.parquet`. Attach **output3** and
**output5** via *+ Add Input → Your Work*.

## 0. Load, and index everything as integers

In [ ]:
import os, re, math, itertools, gc
import pandas as pd, numpy as np

WORK = "/kaggle/working"; os.makedirs(WORK, exist_ok=True)
TARGET = dict(users=7_507, pois=80_962, cats=436, ck=1_214_631)
CITY_CENTRE = {"New York": (40.707864, -73.905237), "Chicago": (41.826546, -87.641298),
               "Los Angeles": (34.000002, -118.250001)}

def find(pat, roots=("/kaggle/input", WORK)):
    hits = []
    for root in roots:
        if not os.path.isdir(root): continue
        for dp, _, fns in os.walk(root):
            if "__MACOSX" in dp: continue
            for fn in fns:
                if re.search(pat, fn, re.I) and not fn.startswith("._"):
                    hits.append(os.path.join(dp, fn))
    return sorted(hits)

def load(pat, what, **kw):
    p = find(pat); assert p, f"{what} not found -- attach output3 and output5 via + Add Input"
    print("loading", p[0])
    return pd.read_parquet(p[0]) if p[0].endswith(".parquet") else pd.read_csv(p[0], **kw)

ck   = load(r"llmgpr_checkins_A\.(parquet|csv)$", "check-ins", dtype=str)
pois = load(r"llmgpr_pois_xy\.(parquet|csv)$",    "POI coordinates", dtype={"venue_id": str})
pois["lat"] = pd.to_numeric(pois["lat"], errors="coerce")
pois["lon"] = pd.to_numeric(pois["lon"], errors="coerce")
pois = pois.dropna(subset=["lat", "lon"]).drop_duplicates("venue_id").reset_index(drop=True)
print(f"{len(ck):,} check-ins | {ck['user_id'].nunique():,} users | {len(pois):,} venues")

# integer-code everything once; every sweep below is then bincount arithmetic
uid, users_u = pd.factorize(ck["user_id"], sort=False)
vid_all, venues_u = pd.factorize(pois["venue_id"], sort=False)
vpos = pd.Series(np.arange(len(venues_u)), index=venues_u)
vid = ck["venue_id"].map(vpos).to_numpy()
assert not np.isnan(vid).any(), "a check-in references a venue with no coordinates"
vid = vid.astype(np.int64)
NU, NV = len(users_u), len(venues_u)

vcat, cats_u = pd.factorize(pois["category"].fillna("?"), sort=False)
vcity, city_u = pd.factorize(pois["venue_id"].map(
    dict(zip(ck["venue_id"], ck["city"]))).fillna("?"), sort=False)

def haversine_km(lat, lon, lat0, lon0):
    R = 6371.0088
    p, p0 = np.radians(lat), math.radians(lat0)
    dp, dl = p - p0, np.radians(lon - lon0)
    return 2 * R * np.arcsin(np.sqrt(np.sin(dp / 2) ** 2 +
                                     np.cos(p) * math.cos(p0) * np.sin(dl / 2) ** 2))

# distance from each venue to ITS OWN city centre (per-city radii need this, not the min)
vkm = np.full(NV, np.inf)
for ci, c in enumerate(city_u):
    if c not in CITY_CENTRE: continue
    la, lo = CITY_CENTRE[c]; m = vcity == ci
    vkm[m] = haversine_km(pois["lat"].to_numpy()[m], pois["lon"].to_numpy()[m], la, lo)
print("cities:", dict(zip(city_u, np.bincount(vcity, minlength=len(city_u)))))
print("venues with a finite distance:", f"{np.isfinite(vkm).sum():,} / {NV:,}")

full_counts = np.bincount(uid, minlength=NU)          # every user's whole 3-city history
print(f"full-history totals: {full_counts.sum():,} check-ins over {(full_counts > 0).sum():,} users")

## 1. The catalogue as a function of radius — including its category count

`#POIs` and `#categories` under the catalogue reading depend on region extent alone, so they can
be tabulated before any user filter enters. The `cats` column here is the one test D got wrong.

In [ ]:
print(f"{'R km':>6}{'catalogue POIs':>16}{'catalogue cats':>16}{'in-region ck':>14}"
      f"{'vs their 80,962':>17}{'vs their 436':>14}")
print("-" * 83)
for R in (5, 8, 9, 10, 11, 12, 14, 16, 20, 25, 30, 40, 999):
    keep = vkm <= R
    npoi = int(keep.sum()); ncat = len(np.unique(vcat[keep]))
    nck = int(keep[vid].sum())
    print(f"{R if R != 999 else 'all':>6}{npoi:>16,}{ncat:>16,}{nck:>14,}"
          f"{npoi / TARGET['pois']:>16.2f}x{ncat / TARGET['cats']:>13.2f}x")
print("-" * 83)
print("\nIf 436 only appears at the widest extent, the category column belongs to the full box")
print("and the 10 km catalogue contradicts it -- that is the tension to report either way.")

## 2. The four accounting conventions

Two independent binary choices, never distinguished in their §4.1:

| | threshold measured on | `#check-ins` counted over |
|---|---|---|
| `in/in` | in-region check-ins | in-region check-ins (this is test D) |
| `in/full` | in-region check-ins | the selected users' full histories |
| `full/in` | the full history | in-region check-ins |
| `full/full` | the full history | the full history |

`in/full` is the one predicted by the 1.99x.

In [ ]:
def match4(got, t=TARGET):
    return float(np.mean([min(got[k], t[k]) / max(got[k], t[k]) for k in ("users", "pois", "cats", "ck")]))

def evaluate(keep_v, Tu, mode):
    """keep_v: boolean mask over venues. mode: 'in/in','in/full','full/in','full/full'."""
    thr_src, ck_src = mode.split("/")
    inreg = keep_v[vid]
    reg_counts = np.bincount(uid[inreg], minlength=NU)
    basis = reg_counts if thr_src == "in" else full_counts
    sel = basis >= Tu
    if not sel.any(): return None
    n_ck = int(reg_counts[sel].sum()) if ck_src == "in" else int(full_counts[sel].sum())
    if n_ck == 0: return None
    return dict(users=int(sel.sum()), pois=int(keep_v.sum()),
                cats=int(len(np.unique(vcat[keep_v]))), ck=n_ck)

MODES = ("in/in", "in/full", "full/in", "full/full")
TUS = (5, 10, 15, 20, 25, 30, 40, 50, 55, 58, 60, 62, 64, 66, 68, 70, 75,
       80, 90, 100, 125, 150, 200)
RADII = (5, 8, 9, 10, 11, 12, 14, 16, 20, 25, 30, 40, 999)

rows = []
for R, mode in itertools.product(RADII, MODES):
    keep = vkm <= R
    for Tu in TUS:
        g = evaluate(keep, Tu, mode)
        if g: rows.append((match4(g), R, mode, Tu, g))
rows.sort(key=lambda r: -r[0])

print(f"top 15 of {len(rows)} (radius x mode x threshold)")
hdr = (f"{'match':>7}{'R km':>7}{'mode':>11}{'Tu':>5}{'users':>9}{'POIs':>9}"
       f"{'cats':>6}{'check-ins':>12}{'ck/user':>9}")
print(hdr); print("-" * len(hdr))
for m, R, mode, Tu, g in rows[:15]:
    print(f"{m:>7.3f}{(R if R != 999 else 'all'):>7}{mode:>11}{Tu:>5}{g['users']:>9,}"
          f"{g['pois']:>9,}{g['cats']:>6}{g['ck']:>12,}{g['ck'] / g['users']:>9.1f}")
print("-" * len(hdr))
print(f"{'TARGET':>7}{'':>7}{'':>11}{'':>5}{TARGET['users']:>9,}{TARGET['pois']:>9,}"
      f"{TARGET['cats']:>6}{TARGET['ck']:>12,}{TARGET['ck'] / TARGET['users']:>9.1f}")

print("\nbest per mode:")
for mode in MODES:
    b = max((r for r in rows if r[2] == mode), key=lambda r: r[0], default=None)
    if b: print(f"  {mode:<10} {b[0]:.3f}  at R={b[1] if b[1] != 999 else 'all'}, Tu>={b[3]}")

## 3. Per-city radii

Their LA/Chicago slices are ~4x under-weight relative to NYC, so one shared radius is probably
wrong. Coordinate grid over the three radii, at the winning accounting mode.

In [ ]:
BEST_MODE = max(MODES, key=lambda mo: max((r[0] for r in rows if r[2] == mo), default=0))
print("carrying forward mode:", BEST_MODE)
if BEST_MODE == "full/full":
    print("\n!! In full/full the radius touches ONLY #POIs and #categories -- #users and")
    print("   #check-ins are computed from full histories and are invariant to it. So this")
    print("   grid fits 3 free radii against essentially ONE scalar (their 80,962).")
    print("   Treat a per-city win as a FIT, not as evidence about their regions; the")
    print("   single-radius result below carries one free parameter and is the real finding.")
CITY_R = (5, 8, 10, 12, 15, 20, 30, 999)
ci_of = {c: i for i, c in enumerate(city_u)}
order = [c for c in ("New York", "Los Angeles", "Chicago") if c in ci_of]

best = None
for combo in itertools.product(CITY_R, repeat=len(order)):
    keep = np.zeros(NV, dtype=bool)
    for c, R in zip(order, combo):
        keep |= (vcity == ci_of[c]) & (vkm <= R)
    if not keep.any(): continue
    for Tu in TUS:
        g = evaluate(keep, Tu, BEST_MODE)
        if g is None: continue
        m = match4(g)
        if best is None or m > best[0]: best = (m, combo, Tu, g)

m, combo, Tu, g = best
desc = ", ".join("%s=%s" % (c, "all" if R == 999 else R) for c, R in zip(order, combo))
print("\nbest per-city radii (%s), Tu >= %d, mode %s  -> match %.3f" % (desc, Tu, BEST_MODE, m))
print(f"{'column':<22}{'ours':>12}{'theirs':>12}{'ratio':>9}")
print("-" * 55)
for k, lab in (("users", "users"), ("pois", "POIs (catalogue)"),
               ("cats", "categories"), ("ck", "check-ins")):
    print(f"{lab:<22}{g[k]:>12,}{TARGET[k]:>12,}{g[k] / TARGET[k]:>9.2f}x")
print(f"{'check-ins per user':<22}{g['ck'] / g['users']:>12.1f}"
      f"{TARGET['ck'] / TARGET['users']:>12.1f}{(g['ck'] / g['users']) / (TARGET['ck'] / TARGET['users']):>9.2f}x")

## 4. Verdict, and emit the winner

In [ ]:
single = rows[0]
print(f"single radius : {single[0]:.3f}  (R={single[1] if single[1] != 999 else 'all'}, "
      f"{single[2]}, Tu>={single[3]})")
print(f"per-city radii: {m:.3f}")
print(f"prior best (Tu=60/Tp=3 interaction filter, output42): 0.923")
win = max(m, single[0])
print()
if win >= 0.95:
    print("=> PIPELINE RECOVERED. All four columns within a few percent under a single coherent")
    print("   reading: the region fixes the POI catalogue, the users' own histories fix the")
    print("   sequences. Their S4.1 sentence describes neither, but the numbers are theirs.")
elif win >= 0.923:
    print("=> BETTER THAN THE INTERACTION-FILTER READING. Adopt the catalogue convention, state")
    print("   the radius and the accounting explicitly, and footnote that S4.1 does not describe it.")
else:
    print("=> NO IMPROVEMENT. The catalogue reading hits #POIs but cannot hold the other columns")
    print("   at the same time. Keep Tu=60/Tp=3, report our own numbers, footnote the gap.")
    print("   The POI column is then closed for good: it is not jointly reproducible.")

# Emit. Prefer the single radius unless per-city buys something material: it has three
# free parameters against one target, and its winners truncate whole cities (a 5 km Chicago
# guts the catalogue the 500-nearest-candidate sampler is built on).
MATERIAL = 0.02
use_percity = (m - single[0]) >= MATERIAL
print(f"\nper-city gain over single radius: {m - single[0]:+.3f} "
      f"(materiality {MATERIAL:.2f}) -> emitting the "
      + ("per-city" if use_percity else "single-radius") + " reading")
if use_percity:
    keep = np.zeros(NV, dtype=bool)
    for c, R in zip(order, combo): keep |= (vcity == ci_of[c]) & (vkm <= R)
    tag, Tu_w, mode_w = "percity_" + "_".join(str(r) for r in combo), Tu, BEST_MODE
else:
    _, R1, mode_w, Tu_w, _ = single
    keep = vkm <= R1; tag = f"R{R1}"
    print(f"  uniform R = {R1} km, Tu >= {Tu_w}, mode {mode_w}")

inreg = keep[vid]
reg_counts = np.bincount(uid[inreg], minlength=NU)
basis = reg_counts if mode_w.split("/")[0] == "in" else full_counts
rowmask = np.isin(uid, np.where(basis >= Tu_w)[0])
if mode_w.split("/")[1] == "in": rowmask &= inreg
out = ck[rowmask].copy()
cat_pois = pois[keep].copy()
print(f"\nemitting {tag} / {mode_w} / Tu>={Tu_w}: {len(out):,} check-ins, "
      f"{out['user_id'].nunique():,} users, catalogue {len(cat_pois):,} POIs")
for df, stem in ((out, "llmgpr_cat_checkins"), (cat_pois, "llmgpr_cat_catalogue")):
    try: df.to_parquet(f"{WORK}/{stem}.parquet", index=False); print("wrote", f"{stem}.parquet")
    except ImportError: df.to_csv(f"{WORK}/{stem}.csv", index=False); print("wrote", f"{stem}.csv")